# 9 · Capstone: a defensible STAC-to-RustGIS brief

**Question:** can you pass catalog results to a GIS and explain exactly what the map shows?

You will combine the WorldPop Degree of Urbanisation snapshot, a small chart, a provenance note, a RustGIS Web link, and a proposed Ollama question. The result is a metadata coverage brief—not a raster analysis.

![STAC to RustGIS flow](https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-flow.svg)

## Load the evidence package

In [ ]:
import json
from pathlib import Path
items=json.loads(Path('data/worldpop_uga_items.geojson').read_text())['features']
prov=json.loads(Path('data/worldpop_provenance.json').read_text())
assert len(items)==len(prov['item_ids'])
print('Provider:',prov['provider'],'| items:',len(items),'| retrieved:',prov['retrieved_on'])

## Extract only claims supported by metadata

In [ ]:
summary=[{'id':i['id'],'year':i['properties'].get('year'),'bbox':i.get('bbox'),'assets':len(i.get('assets',{}))} for i in items]
for row in summary: print(row)

## Visualize a cautious comparison

In [ ]:
from IPython.display import display, HTML
from viz import bars
display(HTML(bars([r['id'] for r in summary],[r['assets'] for r in summary],'Advertised asset counts',' assets')))

## Export a compact GIS layer

In [ ]:
layer={'type':'FeatureCollection','features':[{'type':'Feature','id':i['id'],'geometry':i['geometry'],'properties':{'id':i['id'],'year':i['properties'].get('year'),'asset_count':len(i.get('assets',{}))}} for i in items]}
Path('capstone_stac_footprints.geojson').write_text(json.dumps(layer,indent=2))
print('Saved',len(layer['features']),'footprints')

## Build the brief and provenance

In [ ]:
brief={
 'title':'WorldPop UGA STAC metadata coverage',
 'claim':'Three catalog Item footprints are available in the saved query snapshot.',
 'not_claimed':'No raster cell values, exposure estimates, or uncertainty estimates were analyzed.',
 'source_query':prov['query_url'],
 'retrieved_on':prov['retrieved_on'],
 'item_ids':[i['id'] for i in items],
 'next_step':'Inspect collection license/provider, then open GeoJSON in RustGIS.'
}
Path('capstone_brief.json').write_text(json.dumps(brief,indent=2))
print(json.dumps(brief,indent=2))

## Open the real application

Download `capstone_stac_footprints.geojson` and use the file picker in [RustGIS Web](https://web.rustgis.app/). Alternatively, [open the hosted snapshot directly](https://web.rustgis.app/?blank&data=https%3A%2F%2Fjltobias.github.io%2FJupyterLite-RustGIS%2Fdata%2Fworldpop_uga_items.geojson). Inspect the attributes and symbolize `year`. The direct URL opens the hosted snapshot, not your newly generated local export.

## Ask the assistant carefully

In native RustGIS with Ollama configured, try: *“Summarize the three STAC footprint features and propose a style by year. State explicitly that the polygons show catalog coverage rather than Degree of Urbanisation raster cell values. Show the commands before any file or network action.”* Review every action and result.

## Self-check

- Can someone reproduce the query from your brief?
- Is the provider and retrieval date present?
- Are metadata and raster values kept separate?
- Did you inspect license and asset access before downloading?
- Would another reader know which steps happened in JupyterLite and which happened in RustGIS?

Continue with the [STAC Browser Explorer](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/) for another bounded search, or use the [native RustGIS notebook](https://github.com/jltobias/JupyterLite-RustGIS/blob/main/notebooks/04_native_rustgis.ipynb) for Python `Session` analysis. [Read the book](https://jltobias.github.io/JupyterLite-RustGIS/book/index.html) for the complete route.

**Sources:** [RustGIS](https://github.com/opengeos/RustGIS), [RustGIS web URL parameters](https://github.com/opengeos/RustGIS/blob/main/docs/installation.md#web-app), [your STAC Browser](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/), [STAC specification](https://stacspec.org/). All hosted data here is public teaching material. Check the provider's license before reusing STAC assets.